# One reporting window across several banks

Use three **offline interface samples** for eight aligned quarters, 2024Q3 through 2026Q2. They have different sizes and business models; they are not a matched peer group or a population suitable for causal inference. Every selected bank has one row per requested quarter, even when a live filing is absent.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

try:
    installed = importlib.metadata.version("us-bank-data") == "0.2.0"
except importlib.metadata.PackageNotFoundError:
    installed = False
if not installed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "us-bank-data==0.2.0"])


In [ ]:
from usbankdata import load_sample_cohort

data = load_sample_cohort()
assert len(data.rows) == 24
assert data.certs == [628, 3510, 58665]
assert data.document["requested_dates"] == sorted(data.document["requested_dates"])
print(data.document["coverage"])


## Compute a change only when both observations exist

Do not substitute an older filing for a missing requested quarter. A missing source field is different from an absent filing, recorded by `report_available`. This example calculates deposit changes separately from the immutable reported values.


In [ ]:
for cert in data.certs:
    bank_rows = [row for row in data.rows if row["cert"] == cert]
    start, end = bank_rows[0]["deposits_usd"], bank_rows[-1]["deposits_usd"]
    change = None if start is None or end is None else end - start
    print(cert, bank_rows[-1]["bank_name"], "deposit change USD:", change)


In [ ]:
from pathlib import Path
from usbankdata import BankCohort

output = Path("outputs/cohort-history")
data.export(output / "bank-quarter-panel.csv")
data.export(output / "bank-quarter-panel.json")
assert BankCohort.load(output / "bank-quarter-panel.json").dataset_id == data.dataset_id
print(data.dataset_id)


## Choose your own live banks

The following is optional and is intentionally not executed by this offline lesson:

```python
from usbankdata import panel
live = panel(certs=[628, 3510, 58665], through="2026Q2", quarters=8)
```

A live call supports up to 100 explicit CERTs and 40 quarters, with bounded pagination and retries. It retrieves current source revisions and identity labels. It does not silently filter out inactive banks. Select a cohort from a single-quarter snapshot first if state or size matters, and keep that snapshot selection manifest alongside the historical result. CERT identifies a bank institution, not its listed parent or ticker.


## Exercise and solution direction

Add a missing-quarter check before calculating percentage growth. A valid solution returns missing when the first observation is absent or nonpositive, and keeps that result distinct from zero growth. Original source: FDIC BankFind Suite; BanksOnSale sample compilation and metadata, surfaced by US Bank Data. [Contract](https://github.com/anshuman-pandey/us-bank-data/blob/v0.2.0/docs/COHORTS.md).
